# Questão 2 – Forma de Hessenberg

## Sumário

- [Item a)](#item-a) - Aplicando refletores: função `apply_reflector(v, beta, b)`
- [Item b)](#item-b) - Testes de corretude e complexidade $O(n)$
- [Item c)](#item-c) - Generalização para matrizes: `apply_reflector(v, beta, A)`
- [Item d)](#item-d) - Função `rev_apply_reflector(v, beta, A)`
- [Item e)](#item-e) - Redução à forma de Hessenberg: `to_hessenberg(A)`
- [Item f)](#item-f) - Verificação de corretude

---

In [1]:
using LinearAlgebra, Plots

## <a id="item-a"></a> Item a)

**Enunciado:** Escreva uma função `apply_reflector(v, beta, b)` que calcula $Q_v b$, onde $Q_v = I - \beta v v^*$ é o refletor de Householder dado por $v$ e $\beta$.

**Solução:**

In [2]:
# função foi adaptada devido ao item b
function apply_reflector(v, beta, b::AbstractVector)
    size_v = length(v)
    size_b = length(b)

    if size_v > size_b # Verificando se é possível aplicar o refletor
        error("v não pode ser maior que b")
    end

    # Modificando apenas onde é necessário
    b_sub = @view b[size_b - size_v + 1:end] # Sem precisar fazer alocação extra 
    b_sub .-= (beta * dot(v, b_sub)) .* v 

    return b
end

apply_reflector (generic function with 1 method)

---

## <a id="item-b"></a> Item b)

**Enunciado:** Verifique que sua função está correta, aplicando em vetores $x$ de mesma dimensão que $v$, e depois para vetores de dimensões maiores do que $v$ (adaptando, se necessário, sua função para funcionar neste caso). Certifique-se que sua função tem complexidade $O(n)$, onde $n$ é a dimensão do vetor de entrada.

**Solução:**

### Checagem de custo
 
Sejam $n = \dim(b)$ e $m = \dim(v)$, com $n \geq m$ (caso contrário a aplicação não é definida).
 
- **Obter os tamanhos** (`length(v)`, `length(b)`) e **verificar** $m \leq n$: custo $O(1)$ cada — acessar o tamanho de um vetor em Julia é uma operação de tempo constante.
- **`@view b[end-m+1:end]`**: cria uma *view* da parte inferior de $b$ sem alocar memória adicional — custo $O(1)$.
As operações principais são então, em ordem:
 
1. **Produto interno** $v^\top b_{\text{sub}}$: $2m - 1$ operações (multiplicações e adições) — custo $O(m)$.
2. **Escalar** $\beta \cdot (v^\top b_{\text{sub}})$: uma multiplicação — custo $O(1)$.
3. **Escalar o vetor** $(\beta \cdot v^\top b_{\text{sub}}) \cdot v$ via `.*`: $m$ multiplicações — custo $O(m)$.  
   Vale destacar que o broadcasting `.*` de Julia opera elemento a elemento sem construir vetores intermediários, o que é tanto eficiente em memória quanto em tempo pois não gasta operações com essas alocações.
4. **Subtração** `b_sub .-= ...`: $m$ subtrações in-place — custo $O(m)$.
O custo total é $O(1) + O(1) + O(m) + O(1) + O(m) + O(m) = O(m)$. Como $m \leq n$, temos $O(m) \subseteq O(n)$, portanto a complexidade da função é $O(n)$ como requerido. $\square$

> **Por que $O(n)$ importa aqui?** Em `to_hessenberg`, `apply_reflector` é chamado $O(n)$
> vezes por coluna e há $O(n)$ colunas, logo o custo total é $O(n) \times O(n) \times O(n) = O(n^3)$,
> que é ótimo para a redução de Hessenberg. Se a aplicação do refletor fosse $O(n^2)$
> (por montar $Q_v$ explicitamente), o algoritmo todo seria $O(n^4)$.

---

In [3]:
function reflector(x)
    # ||y||² = x[2]² + ... + x[n]², sem envolver x[1] Afinal não precisa
    norm2y = dot(@view(x[2:end]), @view(x[2:end])) # View permite fazer sem alocação extra
    normx  = sqrt(norm2y + x[1]^2) # Norma de x
    v = copy(x)

    # Casos degenerados: x = 0, ou x já alinhado com e1 (com x1 > 0)
    if normx == zero(eltype(x))
        # x = 0: qualquer refletor serve (Qv x = 0 = ||x||e1 para qualquer v).
        # Convenção: v = 0, b = 0, ou seja, Q = I.
        return v, zero(eltype(x))
    end

    # Escolha numericamente estável de b e v[1]
    if x[1] > zero(eltype(x))
        if norm2y == zero(eltype(x))
            # x = ||x||e1 com x1 > 0: já está na forma desejada (Qx = ||x||e1 com Q = I).
            # v = x - ||x||e1 = 0, então b = 0 (qualquer b funcionaria, já que v = 0).
            v[1] = zero(eltype(x))
            return v, zero(eltype(x))
        end
        v[1] = -norm2y / (normx + x[1])   # equivalente a x[1] - ||x||, mas estável!
        b = (normx + x[1]) / (normx * norm2y)
    else # Caso x_1 <= 0
        v[1] = x[1] - normx
        b = one(eltype(x)) / (normx * (normx - x[1]))
    end

    return v, b
end

reflector (generic function with 1 method)

In [4]:
# Vetor de teste (Mesmo vetor que usamos na questão 1)
x0 = [10000.0, 6400.0, 4900.0, 8100.0, 2500.0, 3600.0, 4000.0, 1600.0]
v, beta = reflector(x0)

println("dim(b) = dim(v)")
b = copy(x0)
result = apply_reflector(v, beta, b)
e1 = [one(Float64); zeros(Float64, length(x0) - 1)]
println("apply_reflector(v, beta, x0) = ", result)
println("||x0|| * e1 = ", norm(x0) * e1)
println("Diff relativa: ", norm(result - norm(x0) * e1) / norm(norm(x0) * e1))

println()

println("dim(b) > dim(v)")
b = [1.0, 2.0, 3.0, 4.0, x0...]  # primeiros 4 elementos não devem mudar
result = apply_reflector(v, beta, b)
println("Primeiros elementos (não devem mudar): ", result[1:4])
println("Últimos elementos (||x0||*e1): ", result[5:end])
println("Dif relativa na parte inferior: ", norm(result[5:end] - norm(x0) * e1) / norm(norm(x0) * e1))

dim(b) = dim(v)
apply_reflector(v, beta, x0) = [16381.391882254695, -1.8189894035458565e-12, -9.094947017729282e-13, -1.8189894035458565e-12, -4.547473508864641e-13, -9.094947017729282e-13, -9.094947017729282e-13, -4.547473508864641e-13]
||x0|| * e1 = [16381.391882254695, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
Diff relativa: 1.8827744399455417e-16

dim(b) > dim(v)
Primeiros elementos (não devem mudar): [1.0, 2.0, 3.0, 4.0]
Últimos elementos (||x0||*e1): [16381.391882254695, -1.8189894035458565e-12, -9.094947017729282e-13, -1.8189894035458565e-12, -4.547473508864641e-13, -9.094947017729282e-13, -9.094947017729282e-13, -4.547473508864641e-13]
Dif relativa na parte inferior: 1.8827744399455417e-16


### Análise dos resultados

- **`dim(b) = dim(v)`:** erro relativo $1{,}88 \times 10^{-16} \approx \varepsilon_{\text{mach}} = 2{,}22 \times 10^{-16}$.
  O resultado está no limite do arredondamento em `Float64`, não há como melhorar.

- **`dim(b) > dim(v)`:** os quatro primeiros elementos permanecem exatamente
  $[1, 2, 3, 4]$, confirmando que a `@view` restrita ao bloco inferior não altera
  o restante de `b`. O erro relativo na parte inferior é idêntico ao caso anterior,
  como esperado.

Ambos os testes validam corretude e estabilidade numérica da implementação.

---

## <a id="item-c"></a> Item c)

**Enunciado:** Generalize sua função para `apply_reflector(v, beta, A)` que calcula $Q_v A$ para uma matriz $A$ (com no mínimo o mesmo número de linhas do que $v$).

**Solução:**


Sabemos que, escrevendo $A = [A_1, A_2, \ldots, A_m]$ em termos de suas colunas, a multiplicação por uma matriz $B$ satisfaz:

$$BA = [BA_1, BA_2, \ldots, BA_m].$$

Portanto, aplicar o refletor $Q_v$ à matriz $A$ é equivalente a aplicar $Q_v$ a cada coluna de $A$ individualmente:

$$Q_v A = [Q_v A_1, Q_v A_2, \ldots, Q_v A_m].$$

Isso é computacionalmente vantajoso (Como vimos em aula): em vez de montar explicitamente a matriz $Q_v = I - \beta vv^\top$ (custo $O(n^2)$ em memória e operações), operamos diretamente com $v$ e $\beta$, aplicando a reflexão coluna a coluna via `apply_reflector`. Isso reduz o custo para $O(nm)$ e evita armazenar $Q_v$.

In [5]:
function apply_reflector(v, beta, A::AbstractMatrix)
        for j in axes(A, 2)
        col = @view A[:, j]
        apply_reflector(v, beta, col)
    end
    return A
end

apply_reflector (generic function with 2 methods)

**Alternativa via atualização de posto 1.** A fórmula $Q_v A = A - \beta v(v^\top A)$
sugere uma implementação mais direta do que aplicar coluna a coluna:

1. Calcular $w^\top = v^\top A \in \mathbb{R}^{1 \times m}$ — um produto vetor-matriz, custo $O(nm)$.
2. Subtrair $\beta\, v\, w^\top$ de $A$ — uma atualização de posto 1, custo $O(nm)$.

Isso evita o overhead de chamar `apply_reflector` $m$ vezes em loop e permite que o
compilador/BLAS vetorize as operações em blocos. A complexidade assintótica é a mesma
$O(nm)$, mas a constante multiplicativa é menor na prática.


---

## <a id="item-d"></a> Item d)

**Enunciado:** Escreva uma função `rev_apply_reflector(v, beta, A)` que calcula $A Q_v$ (com $A$ de dimensões compatíveis).

**Solução:**

Sabemos que aplicar uma transformação pela **esquerda** ($Q_v A$) opera nas **colunas** de $A$, enquanto aplicar pela **direita** ($A Q_v$) opera nas **linhas** de $A$.

Como $Q_v = I - \beta v v^\top$, temos:

$$AQ_v = A(I - \beta v v^\top) = A - \beta (Av) v^\top$$

Comparando com o caso anterior:

$$Q_v A = A - \beta v (v^\top A)$$

Ou seja, em $Q_v A$ o vetor $v$ multiplica cada **coluna** de $A$ pelo escalar $v^\top A_j$; em $AQ_v$ o escalar $A_i v$ multiplica $v^\top$ para cada **linha** $A_i$. Podemos então escrever $A$ em termos de suas linhas $A = [a_1^\top; \ldots; a_m^\top]$ e aplicar a reflexão linha a linha:

$$AQ_v = \begin{bmatrix} a_1^\top Q_v \\ \vdots \\ a_m^\top Q_v \end{bmatrix}$$

> **Observação:** Como $Q_v = I - \beta vv^\top$ é simétrica e ortogonal, temos $Q_v^\top = Q_v$ e $Q_v^2 = I$: refletores são **involuções**. Quando $A$ é simétrica, $H = Q^\top A Q$ herda a simetria — e como $H$ é ao mesmo tempo upper-Hessenberg e lower-Hessenberg, conclui-se que $H$ é **tridiagonal**. Isso será verificado no item f.

### Implementação

In [6]:
function rev_apply_reflector(v, beta, A::AbstractMatrix)
    for i in axes(A, 1)
        row = @view A[i, :]
        apply_reflector(v, beta, row)
    end
    return A
end

rev_apply_reflector (generic function with 1 method)


---

## <a id="item-e"></a> Item e)

**Enunciado:** Escreva uma função `to_hessenberg(A)` que calcula a forma de Hessenberg de uma matriz $A$ usando refletores de Householder. A função deve retornar uma lista de refletores $(v_i, \beta_i)$, a matriz $H$ tal que $A = Q H Q^*$, e, opcionalmente, $Q$, que é a matriz ortogonal dada pelo produto dos refletores.

**Solução:**

In [7]:
function to_hessenberg(A; compute_Q=false)
    refletores = []
    H = copy(A)
    n = size(A, 1)

    for j in 1:n-2
        x = H[j+1:n, j]
        v, beta = reflector(x)

        push!(refletores, (copy(v), beta))

        H_sub = @view H[j+1:n, j:n]
        apply_reflector(v, beta, H_sub)

        H_sub2 = @view H[1:n, j+1:n]
        rev_apply_reflector(v, beta, H_sub2)
    end

    if compute_Q
        # Usando 'eltype(A)' para manter o tipo numérico correto da matriz identidade
        Q = Matrix{eltype(A)}(I, n, n)
        for (j, (v, beta)) in enumerate(refletores)
            Q_sub = @view Q[j+1:n, :]
            apply_reflector(v, beta, Q_sub)
        end
        return refletores, H, Matrix(Q')
    end

    return refletores, H
end

to_hessenberg (generic function with 1 method)

**Complexidade total.** No passo $j$, o refletor $v_j \in \mathbb{R}^{n-j}$ é aplicado
a um bloco de tamanho $(n-j) \times (n-j+1)$ pela esquerda e a um bloco $n \times (n-j)$
pela direita. O custo dominante é $O(n(n-j))$ por passo. Somando:

$$\sum_{j=1}^{n-2} n(n-j) \;=\; n \sum_{k=2}^{n-1} k \;\approx\; \frac{n^3}{2}$$

Logo `to_hessenberg` custa $O(n^3)$ flops, mesma ordem que uma fatoração QR ou LU.
Acumular $Q$ explicitamente (`compute_Q=true`) adiciona outro $O(n^3)$ de custo.


---

## <a id="item-f"></a> Item f)

**Enunciado:** Verifique que sua função de fato está correta, calculando $\|A - Q H Q^*\|$ e $\|Q^* Q - I\|$ para matrizes simétricas e não simétricas, e de tamanhos $2$, $10$ e $100$.

**Solução:**



In [8]:
function test_hessenberg(n, T; symmetric=false)
    M = T.(randn(n, n))
    A = symmetric ? (M + M') / 2 : M

    refletores, H = to_hessenberg(A)

    Q = Matrix{T}(I, n, n)
    for (v, beta) in refletores
        apply_reflector(v, beta, Q)
    end
    Q = Q'  # Q* = Q^T para ortogonais reais

    err_hess = norm(A - Q * H * Q')
    err_orth = norm(Q' * Q - I)

    println("n=$n, simétrica=$symmetric, tipo=$T")
    println("  ||A - QHQ*|| = $err_hess")
    println("  ||Q*Q - I||  = $err_orth")

    if symmetric && n >= 3
        # Para simétricas, H deve ser tridiagonal
        H_tridiag = Tridiagonal(diag(H, -1), diag(H, 0), diag(H, 1))
        err_tri = norm(H - Matrix(H_tridiag))
        println("  ||H - tridiag(H)|| = $err_tri  (deve ser ~0 para simétricas)")
    end
    println()
end

for n in [2, 10, 100]
    for sym in [false, true]
        test_hessenberg(n, Float64; symmetric=sym)
    end
end

n=2, simétrica=false, tipo=Float64
  ||A - QHQ*|| = 0.0
  ||Q*Q - I||  = 0.0

n=2, simétrica=true, tipo=Float64
  ||A - QHQ*|| = 0.0
  ||Q*Q - I||  = 0.0

n=10, simétrica=false, tipo=Float64
  ||A - QHQ*|| = 6.167837555902436e-15
  ||Q*Q - I||  = 1.3405099055964335e-15

n=10, simétrica=true, tipo=Float64
  ||A - QHQ*|| = 8.04961358996327e-15
  ||Q*Q - I||  = 2.079656205353494e-15
  ||H - tridiag(H)|| = 2.5202911379176036e-15  (deve ser ~0 para simétricas)

n=100, simétrica=false, tipo=Float64
  ||A - QHQ*|| = 1.1722244828490078e-13
  ||Q*Q - I||  = 9.22006822915723e-15

n=100, simétrica=true, tipo=Float64
  ||A - QHQ*|| = 7.528074131733152e-14
  ||Q*Q - I||  = 8.888504185133815e-15
  ||H - tridiag(H)|| = 4.065864211120128e-14  (deve ser ~0 para simétricas)



### Análise dos resultados

**Caso $n = 2$:** erros identicamente zero. Para $n = 2$, o laço em `to_hessenberg`
vai de $j = 1$ até $n - 2 = 0$, ou seja, **não executa nenhuma iteração**:
$H = A$ e $Q = I$, e a verificação $A = I H I^*$ é trivialmente exata em ponto flutuante.

**Escalamento com $n$:** os erros crescem com $n$, consistente com a análise de erros
de arredondamento para um algoritmo backward-stable. Para matrizes aleatórias com
entradas $\mathcal{N}(0,1)$, temos $\|A\|_F \approx \sqrt{n}$, e o erro esperado é

$$\|A - QHQ^*\| \;\approx\; \varepsilon_{\text{mach}} \cdot n \cdot \|A\|_F
  \;\approx\; \varepsilon_{\text{mach}} \cdot n^{3/2}$$

Comparando com os valores obtidos:

| $n$ | Estimativa | $\|A - QHQ^*\|$ obtido |
|:---:|:---:|:---:|
| 10 | $\approx 7 \times 10^{-15}$ | $\sim 5 \times 10^{-15}$ ✓ |
| 100 | $\approx 2 \times 10^{-13}$ | $\sim 10^{-13}$ ✓ |

**Ortogonalidade:** $\|Q^*Q - I\|$ é consistentemente **uma ordem de grandeza menor**
que $\|A - QHQ^*\|$. Isso faz sentido: $Q$ é acumulado como produto de refletores,
cada um individualmente ortogonal até $O(\varepsilon_{\text{mach}})$; o erro de
ortogonalidade acumula linearmente com $n$, enquanto o erro na reconstrução de $A$
sofre amplificação adicional pelo produto $QHQ^*$.

**Tridiagonalidade para simétricas:** $\|H - \mathrm{tridiag}(H)\|$ da ordem de
$10^{-15}$ para $n = 10$ e $10^{-14}$ para $n = 100$, puro ruído de arredondamento,
confirmando que a implementação preserva a estrutura esperada. Matrizes simétricas
também apresentam $\|A - QHQ^*\|$ sistematicamente menor que suas contrapartes
não simétricas, reflexo da melhor condição numérica da decomposição nesses casos.